In [29]:
import numpy as np
import pandas as pd
import re
import torch
import torch.nn as nn
import torch.nn.functional as F

class rnn(nn.Module) :
    def __init__ (self, vocab_size, input_size, hidden_size, output_size, pad_idx):
        super(rnn, self).__init__()
        self.embedding = nn.Embedding(vocab_size, input_size, pad_idx)
        self.rnn = nn.RNN(input_size, hidden_size, batch_first=True, num_layers=1, bidirectional=False)
        self.fc = nn.Linear((1+int(0))*hidden_size, output_size)

    def forward (self, x):
        x = self.embedding(x)
        out, _ = self.rnn(x) # replace underscore with a pair of underscores for LSTM
        return self.fc(out) 

GPU_indx = 0
device = torch.device(GPU_indx if torch.cuda.is_available() else 'cpu')

model = rnn(27, 128, 1024, 27, 0).to(device)
model.load_state_dict(torch.load("model.pth"))
model.eval()


rnn(
  (embedding): Embedding(27, 128, padding_idx=0)
  (rnn): RNN(128, 1024, batch_first=True)
  (fc): Linear(in_features=1024, out_features=27, bias=True)
)

In [30]:
CIPHERS = [{0: 14,
  1: 24,
  2: 11,
  3: 23,
  4: 4,
  5: 18,
  6: 17,
  7: 10,
  8: 7,
  9: 5,
  10: 22,
  11: 2,
  12: 25,
  13: 13,
  14: 12,
  15: 21,
  16: 20,
  17: 26,
  18: 16,
  19: 3,
  20: 8,
  21: 1,
  22: 0,
  23: 9,
  24: 19,
  25: 6,
  26: 15},
 {0: 14,
  1: 0,
  2: 22,
  3: 11,
  4: 23,
  5: 8,
  6: 1,
  7: 24,
  8: 17,
  9: 12,
  10: 7,
  11: 6,
  12: 4,
  13: 2,
  14: 18,
  15: 5,
  16: 21,
  17: 3,
  18: 9,
  19: 25,
  20: 26,
  21: 13,
  22: 20,
  23: 15,
  24: 16,
  25: 19,
  26: 10},
 {0: 20,
  1: 24,
  2: 22,
  3: 5,
  4: 13,
  5: 19,
  6: 9,
  7: 26,
  8: 4,
  9: 3,
  10: 7,
  11: 23,
  12: 8,
  13: 17,
  14: 15,
  15: 10,
  16: 2,
  17: 0,
  18: 18,
  19: 21,
  20: 11,
  21: 6,
  22: 14,
  23: 16,
  24: 25,
  25: 12,
  26: 1},
 {0: 7,
  1: 12,
  2: 26,
  3: 18,
  4: 11,
  5: 23,
  6: 13,
  7: 4,
  8: 24,
  9: 10,
  10: 9,
  11: 16,
  12: 14,
  13: 1,
  14: 17,
  15: 15,
  16: 21,
  17: 25,
  18: 3,
  19: 19,
  20: 20,
  21: 8,
  22: 22,
  23: 0,
  24: 6,
  25: 5,
  26: 2},
 {0: 22,
  1: 11,
  2: 17,
  3: 3,
  4: 24,
  5: 26,
  6: 25,
  7: 1,
  8: 7,
  9: 10,
  10: 5,
  11: 12,
  12: 9,
  13: 19,
  14: 20,
  15: 13,
  16: 8,
  17: 23,
  18: 4,
  19: 0,
  20: 18,
  21: 15,
  22: 2,
  23: 14,
  24: 6,
  25: 16,
  26: 21},
 {0: 14,
  1: 21,
  2: 20,
  3: 13,
  4: 25,
  5: 0,
  6: 4,
  7: 1,
  8: 22,
  9: 10,
  10: 3,
  11: 5,
  12: 19,
  13: 24,
  14: 8,
  15: 11,
  16: 16,
  17: 6,
  18: 17,
  19: 2,
  20: 7,
  21: 26,
  22: 12,
  23: 15,
  24: 18,
  25: 9,
  26: 23},
 {0: 10,
  1: 1,
  2: 17,
  3: 2,
  4: 0,
  5: 19,
  6: 23,
  7: 6,
  8: 22,
  9: 14,
  10: 7,
  11: 15,
  12: 24,
  13: 16,
  14: 25,
  15: 4,
  16: 12,
  17: 21,
  18: 20,
  19: 3,
  20: 26,
  21: 5,
  22: 18,
  23: 9,
  24: 13,
  25: 11,
  26: 8},
 {0: 11,
  1: 1,
  2: 22,
  3: 18,
  4: 13,
  5: 12,
  6: 0,
  7: 23,
  8: 16,
  9: 26,
  10: 6,
  11: 20,
  12: 10,
  13: 14,
  14: 8,
  15: 17,
  16: 2,
  17: 19,
  18: 5,
  19: 21,
  20: 3,
  21: 4,
  22: 24,
  23: 15,
  24: 25,
  25: 9,
  26: 7},
 {0: 23,
  1: 10,
  2: 12,
  3: 19,
  4: 24,
  5: 21,
  6: 7,
  7: 25,
  8: 15,
  9: 6,
  10: 20,
  11: 16,
  12: 2,
  13: 0,
  14: 4,
  15: 3,
  16: 9,
  17: 8,
  18: 17,
  19: 14,
  20: 22,
  21: 26,
  22: 18,
  23: 1,
  24: 11,
  25: 5,
  26: 13},
 {0: 7,
  1: 3,
  2: 21,
  3: 20,
  4: 12,
  5: 18,
  6: 14,
  7: 5,
  8: 0,
  9: 24,
  10: 1,
  11: 13,
  12: 26,
  13: 6,
  14: 25,
  15: 2,
  16: 4,
  17: 17,
  18: 19,
  19: 15,
  20: 10,
  21: 8,
  22: 9,
  23: 22,
  24: 11,
  25: 16,
  26: 23}]

In [31]:
def create_cipher(s, idx):
    cipher = CIPHERS[idx]
    lookup = "abcdefghijklmnopqrstuvwxyz "
    s = s.lower()
    s = re.sub(r'[^a-z\s]', '', s)
    s = s[:50]
    encoded = [cipher.get(lookup.index(ch), 0) if ch in lookup else 0 for ch in s]
    return encoded

In [32]:
def test_model(s):
    lookup = "abcdefghijklmnopqrstuvwxyz "
    x = torch.tensor(s, dtype=torch.long).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(x)
        _, predicted = torch.max(outputs, dim=-1)

    decoded = ''.join([lookup[i] for i in predicted.squeeze().tolist() if i < 27])
    print("Decoded:", decoded)

In [33]:
s = "first conception by misdread, Have after-nourishment and life by care; And what was first but fear what might be done, Grows elder now and cares it be not"

In [ ]:
d = create_cipher(s, 4)
test_model(d)

Decoded:  aret conception by misdread have afternourishment
